#**Hybrid Retrieval-Augmented Generation (RAG) with Multi-Query and Reranking**
##**Hybrid Retrieval Complete workflow**

- Hybrid retrieval combining Dense Vector Search (semantic search) and Sparse Keyword Search (lexical search).
- Multi-Query Retrieval
- Reranking

###**0. Install Packages**

In [1]:
# ! pip install langchain

In [2]:
# ! pip install langchain-community

In [3]:
# ! pip install langchain-openai

In [4]:
# ! pip install langchain-chroma

In [5]:
# ! pip install snowballstemmer

In [6]:
# ! pip install sentence-transformers

###**1. Hybrid Retriever**

In [7]:
import chromadb
from chromadb import Schema, SparseVectorIndexConfig, VectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction, OpenAIEmbeddingFunction
from chromadb import Knn, K
from chromadb import Search
from chromadb import Rrf

####**1.1 Setup Chroma Client**

In [8]:
# Create chroma client

# Embedding key
keyFile = open('keys/.azure_openai_embedding_key.txt', 'r')
AZURE_OPENAI_EMBEDDING_API_KEY = keyFile.read().strip()
keyFile.close()

# Chroma key
keyFile = open("keys/.chroma_api_key.txt")
CHROMA_API_KEY = keyFile.read().strip()
keyFile.close()

# Embedding function
embedding_function = OpenAIEmbeddingFunction(
    api_key=AZURE_OPENAI_EMBEDDING_API_KEY,
    api_base="https://gopi-m5ncld8s-eastus2.openai.azure.com/",
    model_name="text-embedding-3-small",
    api_version="2024-12-01-preview"
)

# Create Schema config for Chroma db
schema = Schema()

# Spare vector indexing schema config
schema = schema.create_index(
  key='sparse_vector_key',    
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

# Vector indexing schema 

schema = schema.create_index(
    config=VectorIndexConfig(
        space="cosine",
        embedding_function=embedding_function
    )
)

# Connect to Chroma cloud
client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant="096092a7-6461-4b3a-bbc6-3a9811f7aaa1",
  database='rag_training_db'
)

# Get the collection
collection = client.get_or_create_collection(
  name="friends_collection",
  schema=schema,
)

####**1.2 Set Up Hybrid Retriever**

In [9]:
def hybrid_retriever(query):
    # Sparse embeddings
    sparse_rank = Knn(
      query=query,  # Text query for sparse embeddings
      key="sparse_vector_key",  # Metadata field for sparse vectors
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)
    )
    
    # Dense semantic embeddings
    dense_rank = Knn(
      query=query,  # Text query for dense embeddings
      key="#embedding",          # Default embedding field
      return_rank=True,
      limit=20                 # Only the 20 nearest documents get scored (default limit 16)

    )
    
    # Combine with RRF
    hybrid_rank = Rrf(
      ranks=[dense_rank, sparse_rank],
      weights=[0.7, 0.3],  # 70% semantic, 30% keyword
      k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
    )
    
    # Use in search
    hybrid_search = (Search()
      .rank(hybrid_rank)
      .limit(5)
      .select(K.DOCUMENT, K.SCORE, K.METADATA)
    )
    
    hybrid_results = collection.search(hybrid_search)
    
    return hybrid_results

In [10]:
results = hybrid_retriever("How Ben and Racheal are related?")

In [11]:
print(results.rows()[0])

[{'id': 'cbbe81c0-5e5d-4b8e-9bbc-a77ee713aacf', 'document': "228\n00:15:24,906 --> 00:15:29,138\nActually, we're both the father.\n\n229\n00:15:38,053 --> 00:15:40,521\nOh, Ben! Hey, buddy.\n\n230\n00:15:46,928 --> 00:15:49,396\nPlease tell me you know\nwhich one is our baby.\n\n231\n00:15:51,032 --> 00:15:54,001\nThat one has ducks on his T-shirt\nand this one has clowns.\n\n232\n00:15:54,202 --> 00:15:56,261\nAnd Ben was definitely wearing ducks.\n\n233\n00:15:56,505 --> 00:15:58,166\nOr clowns.", 'metadata': {'sparse_vector_key': SparseVector(indices=[43552621, 48777811, 73978416, 86670560, 106015722, 120071081, 202708381, 233295601, 264233266, 295111310, 318325784, 366837092, 465980904, 516068205, 553793032, 631874927, 634995907, 640494850, 649171654, 697303812, 814103477, 881558406, 936737886, 936994131, 1159720685, 1161364403, 1237113930, 1273489340, 1282029903, 1319819025, 1351999665, 1359641054, 1382082897, 1409699275, 1411441594, 1469784593, 1475817810, 1577650016, 1667382587,

###**2. Multi-Query Retrieval**

####**2.1 Prompt Template for Multi-Query**

In [12]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt to create multiple queries from a given query
template = """
You are a helpful assistant for creating multiple search queries for retrieving relevant documents.
Generate 3 search queries based on the input query provided below.

Input Query: {query}

"""

multi_query_prompt_template = ChatPromptTemplate(
    messages=[
        template
    ]
)

d:\MyData\Work\AI Solution Enablement Program\Labs\projects\hybrid_rag\.env_hybrid_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


####**2.2 Chat Model for Multi-Query**

In [13]:
# Chat Model

from langchain_openai import AzureChatOpenAI

keyFile = open('keys/.azure_openai_api_key.txt', 'r')
AZURE_OPENAI_API_KEY = keyFile.read().strip()
keyFile.close()

chat_model = AzureChatOpenAI(
    azure_endpoint="https://gopi-m5ncld8s-eastus2.openai.azure.com/",
    azure_deployment="Training-gpt-4.1",
    api_version="2024-12-01-preview",
    api_key=AZURE_OPENAI_API_KEY)

####**2.3 Output Parser for Multi-Query**

In [14]:
# List output parser
from pydantic import BaseModel, Field
from typing import List

class MultipleQueryOutput(BaseModel):
    queries: List[str] = Field(default_factory=list, description="A list of search queries")

structured_chat_model = chat_model.with_structured_output(schema=MultipleQueryOutput)


####**2.4 Generator Chain for Multi-Query**

In [15]:
multi_query_chain = multi_query_prompt_template | structured_chat_model

####**2.5 Multe-Query Retriever**

In [16]:
# Multi-query retriever using hybrid retriever
def multi_query_retriever(query):
    # Generate multiple queries
    multi_queries = multi_query_chain.invoke(query)
    
    # Print the generated queries
    print("\nGenerated Queries:")    
    for query in multi_queries.queries:
        print("\t" + query)

    all_results = []
    
    for q in multi_queries.queries:
        results = hybrid_retriever(q)
        all_results.extend(results.rows()[0])
    
    # remove duplicates from results based on id
    unique_results = {result['id']: result for result in all_results}.values()
    
    # Print the unique results (only id and score)
    print("\nUnique Retrieved Context:")
    for result in unique_results:
        print("\tID: " + result['id'] + ", Score: " + str(result['score']))

    return unique_results

In [17]:
results = multi_query_retriever("What is there on the List comparing Rachel and Julie?")


d:\MyData\Work\AI Solution Enablement Program\Labs\projects\hybrid_rag\.env_hybrid_rag\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MultipleQueryOutput(queri...g Rachel versus Julie']), input_type=MultipleQueryOutput])
  return self.__pydantic_serializer__.to_python(



Generated Queries:
	comparison between Rachel and Julie in TV shows or literature
	list of differences and similarities between Rachel and Julie
	summary or discussion comparing Rachel versus Julie

Unique Retrieved Context:
	ID: 725194ea-9d76-4fb8-8865-c7814e1e4c92, Score: -0.016111111
	ID: 32ccd3c5-f4a4-4d8f-952a-e7635bc1512a, Score: -0.016129032
	ID: f86e3063-7552-400c-ad76-bbf6a62d75d2, Score: -0.015482955
	ID: e770db6f-8b53-40b1-b635-d68afc26227e, Score: -0.015952382
	ID: 42b341e5-3f5f-4e44-9a3d-85da7b41e511, Score: -0.01506296
	ID: 4da5bbbb-3b87-4f89-ae59-b979133c3ab8, Score: -0.014993216
	ID: 66b3d76c-3355-4395-a010-298de203e0f5, Score: -0.014697865


In [18]:
len(results)

7

###**3. Reranking**

###**3.1 Initialize Reranker Model**

In [19]:
from langchain_core.documents import Document
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# Import directly from submodule to avoid broken langchain_classic.retrievers.__init__ chain
from langchain_classic.retrievers.document_compressors.cross_encoder_rerank import CrossEncoderReranker

# Initialize the cross-encoder model once
cross_encoder = HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3590.16it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


###**3.2 Reranker function**

In [20]:
# Reranker to compress the document
def reranker(query, results, top_n=3):
    # Convert retrieved rows (dicts) into LangChain Documents
    docs = [
        Document(
            page_content=item["document"],
            metadata={
                "id": item.get("id"),
                "score": item.get("score"),
                **(item.get("metadata") or {}),
            },
        )
        for item in list(results)
    ]

    # Set up the reranker as a compressor
    reranker_compressor = CrossEncoderReranker(model=cross_encoder, top_n=top_n)

    # Rerank documents using compress_documents API
    reranked_docs = reranker_compressor.compress_documents(docs, query=query)

    # Print reranked output (only id and score)
    print("\nReranked Context:")
    for doc in reranked_docs:
        print(f"\tID: {doc.metadata.get('id')}, Score: {doc.metadata.get('score')}")

    return reranked_docs


In [21]:
# Rerank results
query = "What is there on the List comparing Rachel and Julie?"
reranked_docs = reranker(query, results)



Reranked Context:
	ID: f86e3063-7552-400c-ad76-bbf6a62d75d2, Score: -0.015482955
	ID: 725194ea-9d76-4fb8-8865-c7814e1e4c92, Score: -0.016111111
	ID: 4da5bbbb-3b87-4f89-ae59-b979133c3ab8, Score: -0.014993216


##**4. Complete Hybrid RAG Generation**

###**4.1 Multi Query With Reranking**

In [22]:
# Multi Query With Reranking
def multi_query_with_reranking(query):

    # 1. call the multi query retriever
    results = multi_query_retriever(query)

    # 2. rerank the results
    reranked_docs = reranker(query, results)

    return reranked_docs

In [23]:
query = "What is there on the List comparing Rachel and Julie?"
multiqueryresult = multi_query_with_reranking(query)

d:\MyData\Work\AI Solution Enablement Program\Labs\projects\hybrid_rag\.env_hybrid_rag\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MultipleQueryOutput(queri...omparison on the list']), input_type=MultipleQueryOutput])
  return self.__pydantic_serializer__.to_python(



Generated Queries:
	comparison between Rachel and Julie on the list
	what differences are listed between Rachel and Julie
	Rachel vs Julie: items or points of comparison on the list

Unique Retrieved Context:
	ID: 32ccd3c5-f4a4-4d8f-952a-e7635bc1512a, Score: -0.016237315
	ID: 725194ea-9d76-4fb8-8865-c7814e1e4c92, Score: -0.015855532
	ID: f86e3063-7552-400c-ad76-bbf6a62d75d2, Score: -0.016666668
	ID: 4da5bbbb-3b87-4f89-ae59-b979133c3ab8, Score: -0.015135261
	ID: e770db6f-8b53-40b1-b635-d68afc26227e, Score: -0.014833604
	ID: 42b341e5-3f5f-4e44-9a3d-85da7b41e511, Score: -0.014049236

Reranked Context:
	ID: f86e3063-7552-400c-ad76-bbf6a62d75d2, Score: -0.016666668
	ID: 725194ea-9d76-4fb8-8865-c7814e1e4c92, Score: -0.015855532
	ID: 4da5bbbb-3b87-4f89-ae59-b979133c3ab8, Score: -0.015135261


###**4.2 Helper Function for Formatting**

In [24]:
# Helper function to join the retrieved chunks

def format_docs(docs):
    context = ""
    for doc in docs:
        context += doc.page_content
    return context

###**4.3 Runnable for Multi-Query**

In [25]:
# create chain for multi-retriver and reranking
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# runnable for multi query retriever
multi_query_runnable = RunnableLambda(lambda query: multi_query_with_reranking(query))
format_docs_runnable = RunnableLambda(lambda docs: format_docs(docs))
    

###**4.4 Prompt Template for Main Generator**

In [26]:
from langchain_core.prompts import ChatPromptTemplate

# Prompt template for genrator
PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {query}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

###**4.5 Context Chain**

In [27]:
# build the context chain
context_chain = {
    "context": multi_query_runnable | format_docs_runnable,
    "query": RunnablePassthrough()
} 

###**4.6 Output Parser**

In [28]:
# Output Parser
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()


###**4.7 Main Chain**

In [29]:

main_chain = context_chain | prompt_template | chat_model | parser

##**5 Inference**

In [30]:
final_result =  main_chain.invoke("Which episode racheal and julie comes together?")

print("\nFinal output:")
print(final_result)

d:\MyData\Work\AI Solution Enablement Program\Labs\projects\hybrid_rag\.env_hybrid_rag\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=MultipleQueryOutput(queri...el and Julie together']), input_type=MultipleQueryOutput])
  return self.__pydantic_serializer__.to_python(



Generated Queries:
	Friends episode where Rachel and Julie meet
	Which episode does Rachel meet Ross's girlfriend Julie in Friends
	Friends TV show episode featuring Rachel and Julie together

Unique Retrieved Context:
	ID: 1cc8ca19-07ab-4c41-ab52-bcebc75c44fa, Score: -0.01631412
	ID: 725194ea-9d76-4fb8-8865-c7814e1e4c92, Score: -0.016290322
	ID: 9bbd18f4-5d62-4216-924b-abdb4fa19554, Score: -0.01521215
	ID: 32ccd3c5-f4a4-4d8f-952a-e7635bc1512a, Score: -0.0150836725
	ID: 42b341e5-3f5f-4e44-9a3d-85da7b41e511, Score: -0.01440461
	ID: e770db6f-8b53-40b1-b635-d68afc26227e, Score: -0.016014494
	ID: a277405a-990f-496f-b8f6-06f126b811b1, Score: -0.015290322
	ID: d10cd0e6-4aa2-400a-9015-17271a39aea6, Score: -0.014566699

Reranked Context:
	ID: d10cd0e6-4aa2-400a-9015-17271a39aea6, Score: -0.014566699
	ID: 725194ea-9d76-4fb8-8865-c7814e1e4c92, Score: -0.016290322
	ID: 32ccd3c5-f4a4-4d8f-952a-e7635bc1512a, Score: -0.0150836725

Final output:
The episode where Rachel and Julie come together is th